# Nykaa Domain Support Agent — Task 16: Timeouts and Retries

## Objective
Harden the existing Task 7 four-node LangGraph agent while preserving its business logic.

Required demonstrations:
1. A transient failure occurs twice and succeeds on the third attempt using exponential backoff and jitter.
2. A per-node timeout stops a simulated slow node and raises `NodeTimeoutError`.
3. A global timeout cancels the entire graph run when its total execution time exceeds the limit.

The four original nodes remain: `classify_intent`, `retrieve_policy`, `lookup_order`, and `compose_response`.

This notebook reuses Task 5's `nykaa_sentence_chunks` ChromaDB collection and Task 6's `check_order_status()` from `order_lookup.py`.

**Compatibility:** Per-node LangGraph timeouts require `langgraph>=1.2` and async node functions. The global deadline is applied around the full `ainvoke()` call using `asyncio.wait_for()`.

Run from your Nykaa project folder (`C:\\Users\\singh\\Downloads`), where `chroma_db/` and `order_lookup.py` exist.

Official reference: https://docs.langchain.com/oss/python/langgraph/fault-tolerance

## 1. Install / upgrade packages

In [ ]:
%pip install -U "langgraph>=1.2" langgraph-checkpoint-sqlite chromadb sentence-transformers

## 2. Imports and project configuration

In [ ]:
from pathlib import Path
from typing import TypedDict, Optional, Literal
import asyncio
import re
import json
import time
from collections import defaultdict

import chromadb
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy
from langgraph.errors import NodeTimeoutError
from order_lookup import check_order_status

PROJECT_DIR = Path.cwd()
CHROMA_DIR = PROJECT_DIR / "chroma_db"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())
print("ChromaDB directory:", CHROMA_DIR.resolve())

if not CHROMA_DIR.exists():
    raise FileNotFoundError(f"Missing {CHROMA_DIR}. Run Task 3 first.")
if not (PROJECT_DIR / "order_lookup.py").exists():
    raise FileNotFoundError("order_lookup.py is missing from this project folder.")

print("PASS: Project files found.")


## 3. Load existing RAG and order lookup components

In [ ]:
COLLECTION_NAME = "nykaa_sentence_chunks"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 3

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
policy_collection = chroma_client.get_collection(COLLECTION_NAME)
embedding_model = SentenceTransformer(MODEL_NAME)

print("RAG collection:", COLLECTION_NAME)
print("Indexed chunks:", policy_collection.count())
print("Task 6 order lookup:", check_order_status("ORD1001"))

assert policy_collection.count() > 0
assert check_order_status("ORD1001").get("found") is True
print("PASS: Existing RAG and order lookup are available.")


## 4. Define state, transient exception and execution counters

In [ ]:
class SupportState(TypedDict, total=False):
    query: str
    intent: Literal["policy", "order"]
    record_id: Optional[str]
    retrieved_context: list
    order_result: dict
    answer: str
    sources: list
    route_taken: str
    simulate_retry: bool
    simulate_node_timeout: bool
    simulate_global_delay: bool


class TransientServiceError(Exception):
    pass


node_execution_log = []
retry_attempts = defaultdict(int)

ORDER_ID_PATTERN = re.compile(r"\bORD\d{4,}\b", flags=re.IGNORECASE)
ORDER_INTENT_KEYWORDS = {
    "order status", "track my order", "track order",
    "where is my order", "check my order", "order update"
}

print("PASS: State and retry instrumentation configured.")


## 5. Define the four original Task 7 nodes as async functions

In [ ]:
async def classify_intent(state: SupportState) -> dict:
    node_execution_log.append("classify_intent")
    query = state["query"].strip()
    query_lower = query.lower()
    id_match = ORDER_ID_PATTERN.search(query)
    has_order_language = any(term in query_lower for term in ORDER_INTENT_KEYWORDS)

    if id_match or has_order_language:
        intent = "order"
        record_id = id_match.group(0).upper() if id_match else None
    else:
        intent = "policy"
        record_id = None

    print(f"[classify_intent] intent={intent}, record_id={record_id}")
    return {"intent": intent, "record_id": record_id, "route_taken": intent}


async def retrieve_policy(state: SupportState) -> dict:
    node_execution_log.append("retrieve_policy")
    query_embedding = await asyncio.to_thread(
        lambda: embedding_model.encode(
            [state["query"]], normalize_embeddings=True
        )[0].tolist()
    )
    result = await asyncio.to_thread(
        lambda: policy_collection.query(
            query_embeddings=[query_embedding],
            n_results=min(TOP_K, policy_collection.count()),
            include=["documents", "metadatas", "distances"]
        )
    )

    docs = result.get("documents", [[]])[0]
    metas = result.get("metadatas", [[]])[0]
    distances = result.get("distances", [[]])[0]
    contexts = []
    for doc, meta, distance in zip(docs, metas, distances):
        meta = meta or {}
        contexts.append({
            "text": doc,
            "source": meta.get("source", "unknown"),
            "similarity": round(1 - float(distance), 4)
        })

    print(f"[retrieve_policy] retrieved {len(contexts)} chunks")
    return {"retrieved_context": contexts}


async def lookup_order(state: SupportState) -> dict:
    node_execution_log.append("lookup_order")

    if state.get("simulate_node_timeout"):
        print("[lookup_order] Simulating a slow downstream request...")
        await asyncio.sleep(3)

    if state.get("simulate_retry"):
        retry_attempts["lookup_order"] += 1
        attempt = retry_attempts["lookup_order"]
        print(f"[lookup_order] transient-failure test, attempt {attempt}")
        if attempt <= 2:
            raise TransientServiceError(
                f"Simulated temporary failure on attempt {attempt}"
            )

    record_id = state.get("record_id")
    if not record_id:
        result = {
            "found": False,
            "error": "Please provide your order ID (for example, ORD1001)."
        }
    else:
        result = await asyncio.to_thread(check_order_status, record_id)

    print("[lookup_order] lookup completed; found =", result.get("found"))
    return {"order_result": result}


async def compose_response(state: SupportState) -> dict:
    node_execution_log.append("compose_response")

    if state.get("simulate_global_delay"):
        print("[compose_response] Simulating a long-running response step...")
        await asyncio.sleep(5)

    if state.get("intent") == "order":
        result = state.get("order_result", {})
        if not result.get("found"):
            answer = result.get("error", "I could not find that order.")
            sources = []
        else:
            escalation_text = (
                "This order is recommended for human support review."
                if result.get("requires_escalation")
                else "This order does not currently meet the project's escalation threshold."
            )
            answer = (
                f"Order {result['record_id']} has status: {result['status']}. "
                f"Order value: ₹{result['order_value_inr']:,.2f}. "
                f"Escalation score: {result['escalation_score']:.2f}. "
                f"{escalation_text}"
            )
            sources = ["orders dataset (synthetic project data)"]
    else:
        contexts = state.get("retrieved_context", [])
        useful = [
            item for item in contexts
            if item.get("text") and item.get("source") != "unknown"
        ]
        if not useful:
            answer = "I don't know based on the available policy documents."
            sources = []
        else:
            answer = (
                "According to the available project policy documents:\n\n"
                + "\n\n".join(
                    f"[{item['source']}] {item['text']}" for item in useful
                )
            )
            sources = list(dict.fromkeys(item["source"] for item in useful))

    print("[compose_response] final answer created")
    return {"answer": answer, "sources": sources}


print("PASS: Four async Task 7 nodes are defined.")


## 6. Configure exponential-backoff retries and a per-node timeout

`lookup_order` receives:
- Maximum attempts: 3 (including the first attempt)
- Initial interval: 0.2 seconds
- Backoff factor: 2
- Maximum interval: 1 second
- Jitter: enabled
- Retry only `TransientServiceError`
- Per-attempt node timeout: 1 second

The timeout simulation raises `NodeTimeoutError`; it is not included in this custom retry policy.

In [ ]:
lookup_retry_policy = RetryPolicy(
    max_attempts=3,
    initial_interval=0.2,
    backoff_factor=2.0,
    max_interval=1.0,
    jitter=True,
    retry_on=(TransientServiceError,)
)

builder = StateGraph(SupportState)
builder.add_node("classify_intent", classify_intent)
builder.add_node("retrieve_policy", retrieve_policy)
builder.add_node(
    "lookup_order",
    lookup_order,
    retry_policy=lookup_retry_policy,
    timeout=1.0
)
builder.add_node("compose_response", compose_response)

builder.add_edge(START, "classify_intent")
builder.add_conditional_edges(
    "classify_intent",
    lambda state: "order" if state.get("intent") == "order" else "policy",
    {"policy": "retrieve_policy", "order": "lookup_order"}
)
builder.add_edge("retrieve_policy", "compose_response")
builder.add_edge("lookup_order", "compose_response")
builder.add_edge("compose_response", END)

support_agent = builder.compile()

print("PASS: Four-node graph compiled.")
print("Retry policy:", lookup_retry_policy)
print("lookup_order per-node timeout: 1.0 second")
print(support_agent.get_graph().draw_mermaid())


## 7. Baseline test — normal order lookup

In [ ]:
node_execution_log.clear()
retry_attempts.clear()

baseline_result = await support_agent.ainvoke({
    "query": "Please check the status of order ORD1001."
})

print("Route:", baseline_result.get("route_taken"))
print("Order result:", baseline_result.get("order_result"))
print("Answer:", baseline_result.get("answer"))
print("Node execution log:", node_execution_log)

assert baseline_result.get("intent") == "order"
assert baseline_result.get("order_result", {}).get("found") is True
assert baseline_result.get("answer")
print("PASS: Normal order lookup completed.")


## 8. Demonstration A — transient failure recovered by retries

In [ ]:
node_execution_log.clear()
retry_attempts.clear()

retry_result = await support_agent.ainvoke({
    "query": "Please check the status of order ORD1001.",
    "simulate_retry": True
})

print("Lookup attempts:", retry_attempts["lookup_order"])
print("Final order result:", retry_result.get("order_result"))
print("Final answer:", retry_result.get("answer"))
print("Node execution log:", node_execution_log)

assert retry_attempts["lookup_order"] == 3
assert retry_result.get("order_result", {}).get("found") is True
assert retry_result.get("answer")
print("PASS: Retry policy recovered after two transient failures.")


## 9. Demonstration B — per-node timeout fires cleanly

In [ ]:
node_execution_log.clear()
node_timeout_started = time.perf_counter()
node_timeout_caught = False

try:
    await support_agent.ainvoke({
        "query": "Please check the status of order ORD1001.",
        "simulate_node_timeout": True
    })
except NodeTimeoutError as exc:
    node_timeout_caught = True
    elapsed = time.perf_counter() - node_timeout_started
    print("EXPECTED NODE TIMEOUT")
    print("Exception:", type(exc).__name__)
    print("Node:", getattr(exc, "node", None))
    print("Timeout kind:", getattr(exc, "kind", None))
    print("Elapsed wall time:", round(elapsed, 2), "seconds")
    print("Error:", str(exc))
except Exception as exc:
    print("Unexpected exception:", type(exc).__name__, str(exc))
    raise

assert node_timeout_caught, (
    "Expected NodeTimeoutError. Confirm langgraph>=1.2 and async lookup_order."
)
print("PASS: Per-node timeout raised NodeTimeoutError.")


## 10. Demonstration C — global timeout cancels the whole graph

LangGraph's node timeout applies to one node attempt. For a total graph-run deadline, wrap the complete asynchronous `ainvoke()` in `asyncio.wait_for()`.

The response node is delayed for 5 seconds, while the complete graph is allowed only 2 seconds.

In [ ]:
node_execution_log.clear()

GLOBAL_TIMEOUT_SECONDS = 2.0
global_timeout_caught = False
global_started = time.perf_counter()

try:
    await asyncio.wait_for(
        support_agent.ainvoke({
            "query": "Please check the status of order ORD1001.",
            "simulate_global_delay": True
        }),
        timeout=GLOBAL_TIMEOUT_SECONDS
    )
except asyncio.TimeoutError:
    global_timeout_caught = True
    elapsed = time.perf_counter() - global_started
    print("EXPECTED GLOBAL GRAPH TIMEOUT")
    print("Configured graph timeout:", GLOBAL_TIMEOUT_SECONDS, "seconds")
    print("Elapsed wall time:", round(elapsed, 2), "seconds")
    print("Nodes entered before cancellation:", node_execution_log)
except Exception as exc:
    print("Unexpected exception:", type(exc).__name__, str(exc))
    raise

assert global_timeout_caught, "Expected whole-graph asyncio.TimeoutError."
assert "compose_response" in node_execution_log
print("PASS: Global timeout cancelled the graph run.")


## 11. Save Task 16 evidence

In [ ]:
evidence = {
    "task": "Task 16 — Timeouts and retries",
    "graph_nodes": [
        "classify_intent", "retrieve_policy", "lookup_order", "compose_response"
    ],
    "retry_policy": {
        "max_attempts": 3,
        "initial_interval_seconds": 0.2,
        "backoff_factor": 2.0,
        "max_interval_seconds": 1.0,
        "jitter": True,
        "retry_on": "TransientServiceError"
    },
    "retry_demonstration": {
        "attempts": retry_attempts["lookup_order"],
        "succeeded": retry_result.get("order_result", {}).get("found") is True,
        "answer": retry_result.get("answer")
    },
    "per_node_timeout": {
        "node": "lookup_order",
        "configured_timeout_seconds": 1.0,
        "simulated_delay_seconds": 3.0,
        "caught_node_timeout_error": node_timeout_caught
    },
    "global_timeout": {
        "configured_timeout_seconds": GLOBAL_TIMEOUT_SECONDS,
        "simulated_delay_seconds": 5.0,
        "caught_global_timeout": global_timeout_caught
    },
    "acceptance": {
        "retry_recovered": (
            retry_attempts["lookup_order"] == 3
            and retry_result.get("order_result", {}).get("found") is True
        ),
        "per_node_timeout_fired": node_timeout_caught,
        "global_timeout_cancelled_run": global_timeout_caught
    }
}

evidence_path = RESULTS_DIR / "task16_timeout_retry_evidence.json"
evidence_path.write_text(
    json.dumps(evidence, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8"
)

print("Evidence saved:", evidence_path.resolve())
print("\nTASK 16 ACCEPTANCE SUMMARY")
for requirement, passed in evidence["acceptance"].items():
    print(("PASS" if passed else "FAIL") + ":", requirement)

assert all(evidence["acceptance"].values())
print("\nPASS: Task 16 demonstrations completed and evidence saved.")


## 12. Task 16 completion checklist

- [ ] Retry policy specifies attempts, initial interval, max interval, and jitter.
- [ ] Simulated transient failure occurs twice and recovers on attempt 3.
- [ ] Per-node timeout raises a clean `NodeTimeoutError`.
- [ ] Global timeout cancels the entire graph run.
- [ ] Evidence saved to `results/task16_timeout_retry_evidence.json`.

**Important distinction:** the per-node timeout is LangGraph's native async node timeout. The global graph timeout is an outer deadline around the complete `ainvoke()` call, implemented with Python's `asyncio.wait_for()`.

Official reference: https://docs.langchain.com/oss/python/langgraph/fault-tolerance